In [15]:
# Load packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KernelDensity

In [2]:
# Load shots
with open('./data/players_shots_clean.pickle', 'rb') as f:
    players_shots = pickle.load(f)
with open('./data/player_position.pickle', 'rb') as f:
    players_position = pickle.load(f)


In [3]:
# VARIABLES
X_MIN, X_MAX = (-250, 250) 
Y_MIN, Y_MAX = (0, 470)

XX, YY = np.mgrid[0:1:201j, 0:1:201j]
POS = np.vstack([XX.ravel(), YY.ravel()])

In [ ]:
# Functions
BANDWIDTH_GRID = np.logspace(-3, 0, 20)


def select_bandwidth(values, bandwidth_grid=BANDWIDTH_GRID, n_splits=5):
    n_samples = values.shape[0]

    # Construct the folds for cross-validation.
    cv = KFold(
        n_splits=min(n_splits, n_samples),
        shuffle=True,
        random_state=0,
    )
    
    search = GridSearchCV(
        KernelDensity(kernel='gaussian'),
        {'bandwidth': bandwidth_grid},
        cv=cv,
    )
    search.fit(values)
    return search.best_params_['bandwidth']


def estimate_density(df, bandwidth_grid=BANDWIDTH_GRID, n_splits=5):
    X = df.LOC_X.to_numpy(dtype='float')
    Y = df.LOC_Y.to_numpy(dtype='float')
    
    # Rescaling to [0, 1]
    X = (X - X_MIN) / (X_MAX - X_MIN)
    Y = (Y - Y_MIN) / (Y_MAX - Y_MIN)
    
    # Kernel density estimates with cross-validated bandwidth selection.
    values = np.column_stack([X, Y])
    bandwidth = select_bandwidth(values, bandwidth_grid, n_splits)
    kernel = KernelDensity(kernel='gaussian', bandwidth=bandwidth)
    kernel.fit(values)
    density = np.exp(kernel.score_samples(POS.T))
    density[density < 1e-12] = 0
    return np.reshape(density, XX.shape)

In [6]:
# Estimate the density for all the shots (regardless of made and missed shots).
shots_density = players_shots\
    .groupby('PLAYER_ID')\
    .apply(estimate_density, include_groups=False)

In [8]:
shots_density_df = players_shots[['PLAYER_ID', 'PLAYER_NAME']]\
    .drop_duplicates()\
    .join(pd.DataFrame({'DENSITY': shots_density}), on='PLAYER_ID')\
    .reset_index()\
    .drop('index', axis=1)

In [10]:
# Save dataframe
with open('./data/players_shots_density.pickle', 'wb') as f:
    pickle.dump(shots_density_df, f, protocol=pickle.HIGHEST_PROTOCOL)

In [5]:
# Estimate the density for the attempted shots (miss).
players_shots_attempted = players_shots[players_shots.SHOT_MADE_FLAG == 0]

In [6]:
# Estimate the density
shots_density_attempted = players_shots_attempted\
    .groupby('PLAYER_ID')\
    .apply(estimate_density, include_groups=False)

In [7]:
shots_density_attempted_df = players_shots_attempted[['PLAYER_ID', 'PLAYER_NAME']]\
    .drop_duplicates()\
    .join(pd.DataFrame({'DENSITY': shots_density_attempted}), on='PLAYER_ID')\
    .reset_index()\
    .drop('index', axis=1)

In [8]:
# Save dataframe
with open('./data/players_shots_density_attempted.pickle', 'wb') as f:
    pickle.dump(shots_density_attempted_df, f, protocol=pickle.HIGHEST_PROTOCOL)

In [9]:
# Now, we do the same for only the made shots.
players_shots_made = players_shots[players_shots.SHOT_MADE_FLAG == 1]

In [10]:
shots_density_made = players_shots_made\
    .groupby('PLAYER_ID')\
    .apply(estimate_density, include_groups=False)

In [11]:
shots_density_made_df = players_shots[['PLAYER_ID', 'PLAYER_NAME']]\
    .drop_duplicates()\
    .join(pd.DataFrame({'DENSITY': shots_density_made}), on='PLAYER_ID')\
    .reset_index()\
    .drop('index', axis=1) 

In [12]:
# Save dataframe
with open('./data/players_shots_density_made.pickle', 'wb') as f:
    pickle.dump(shots_density_made_df, f, protocol=pickle.HIGHEST_PROTOCOL)